# ollama

> `Chat` over a local [Ollama](https://ollama.com) daemon: install, model pull and `ollama serve` driven from Python, native thinking levels and schema-constrained JSON.

A daemon owns the weights, the KV cache and the GPU, and answers over HTTP. This backend covers
both halves, the wire and the daemon behind it. `Chat('ollama/qwen3:4b')` runs on a machine with no
Ollama on it.

Tools, streaming, HITL and structured output behave as on any backend. See [index](index.html).

In [ ]:
#| default_exp ollama

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, os, time, atexit, platform, shutil, subprocess, tarfile, zipfile
import httpx2 as httpx
from urllib.parse import urlsplit
from fastcore.funccall import get_schema
from fastcore.all import Path, store_attr, L, ifnone, first, listify
import rishi.core
from urai import (Caps, Chat, ChatOpts, Resp, SlidingWindowCallback, StreamSplit, ToolCall, ToolLoopMixin,
                  ToolReminderCallback, UsageCallback, est_tokens, is_path, mk_content, mk_msg, mk_msgs, parse_args, parse_tool_tags, render_prompt,
                  model_caps, resp_text, split_runtime, split_think, thought, truncated, tc_name)

In [ ]:
from fastcore.test import test_eq, test_fail

## Where the daemon is

`OLLAMA_HOST` takes a bare port, a `host:port`, or a full URL. One variable both configures the
daemon and finds it, so `ollama_url` accepts all three spellings.

A loopback daemon must not go through an HTTP proxy. `httpx` reads `$HTTPS_PROXY` from the
environment by default. In a sandbox that sends `127.0.0.1` somewhere with no route back.
`is_local` decides whether a client trusts the environment.

In [ ]:
#| export
#: Where Ollama listens unless `$OLLAMA_HOST` says otherwise.
OLLAMA_HOST = '127.0.0.1:11434'

#: Hostnames that mean this machine, and so must bypass any configured HTTP proxy.
LOCAL_HOSTS = ('127.0.0.1', 'localhost', '::1', '0.0.0.0')

def ollama_url(host=None):
    "Normalize an Ollama host (`:11434`, `host:port`, or a full URL) to a base URL. `None` reads `$OLLAMA_HOST`."
    h = str(host or os.getenv('OLLAMA_HOST') or OLLAMA_HOST).strip()
    if '://' not in h: h = 'http://' + h.lstrip('/')
    u = urlsplit(h)
    return f"{u.scheme}://{u.hostname or '127.0.0.1'}:{u.port or (443 if u.scheme == 'https' else 11434)}"

def is_local(url):
    "Is `url` on this machine? A loopback daemon must not be sent through `$HTTPS_PROXY`."
    return (urlsplit(url).hostname or '') in LOCAL_HOSTS

In [ ]:
test_eq(ollama_url('http://127.0.0.1:11434'), 'http://127.0.0.1:11434')
test_eq(ollama_url('localhost:11434'), 'http://localhost:11434')
test_eq(ollama_url(':11434'), 'http://127.0.0.1:11434')          # the bare-port spelling
test_eq(ollama_url('box.local'), 'http://box.local:11434')       # port defaulted
test_eq(ollama_url('https://gpu.internal'), 'https://gpu.internal:443')

assert is_local(ollama_url()) and is_local(ollama_url('localhost'))
assert not is_local(ollama_url('https://gpu.internal'))          # that one does go through the proxy

## Model ids

Ollama names models `family:tag`. It also serves any GGUF repo on the HuggingFace Hub under an
`hf.co/` prefix. `ollama_model` maps rishi's ids onto that, with `quant` as the tag. The
`Qwen/Qwen3-4B-GGUF` constant `rishi.llama` uses runs here unchanged.

A local `.gguf` file has no address in Ollama's store. Importing one takes a blob upload and an
`/api/create`. rishi says that rather than failing on the wire.

In [ ]:
#| export
#: Ollama library ids that make useful defaults, smallest first. Any id from https://ollama.com/library works.
qwen3_06b  = 'qwen3:0.6b'
qwen3_17b  = 'qwen3:1.7b'
qwen3_4b   = 'qwen3:4b'
gemma3_1b  = 'gemma3:1b'
gemma3_4b  = 'gemma3:4b'
llama32_3b = 'llama3.2:3b'
dflt_model = qwen3_17b   #: what `OllamaChat()` loads when given no model

#: How Ollama addresses the HuggingFace Hub.
HF_HOSTS = ('hf.co/', 'huggingface.co/')

def ollama_model(model=None, quant='Q4_K_M'):
    "A rishi model id as Ollama addresses it: a hub GGUF repo becomes `hf.co/<repo>:<quant>`, an Ollama id passes through."
    m = str(model or dflt_model).strip()
    if is_path(m): raise ValueError(
        f"Ollama serves models from its own store, not a path: {m!r}. Use runtime='llama' for a "
        f"local GGUF file, or import it into Ollama first with `ollama create`.")
    if m.lower().startswith(HF_HOSTS) or '/' not in m: return m
    return f'hf.co/{m}:{quant}' if quant else f'hf.co/{m}'

In [ ]:
test_eq(ollama_model('qwen3:4b'), 'qwen3:4b')                    # an Ollama id is left alone
test_eq(ollama_model('gemma3'), 'gemma3')                          # ...tagless too
test_eq(ollama_model(), dflt_model)
test_eq(ollama_model('Qwen/Qwen3-4B-GGUF'), 'hf.co/Qwen/Qwen3-4B-GGUF:Q4_K_M')     # the rishi.llama id
test_eq(ollama_model('Qwen/Qwen3-4B-GGUF', 'Q8_0'), 'hf.co/Qwen/Qwen3-4B-GGUF:Q8_0')
test_eq(ollama_model('Qwen/Qwen3-4B-GGUF', None), 'hf.co/Qwen/Qwen3-4B-GGUF')      # let Ollama choose
test_eq(ollama_model('hf.co/bartowski/x-GGUF:IQ3_M'), 'hf.co/bartowski/x-GGUF:IQ3_M')
test_fail(lambda: ollama_model('/models/mine.gguf'), contains="runtime='llama'")

## The wire

`OllamaClient` holds the HTTP surface rishi uses. `/api/chat` talks to a model. The management
endpoints list the local store, pull what is missing, and name the models in memory. A refusal
carries a JSON `error` field, and `OllamaError` repeats it.

`/api/pull` and a streamed `/api/chat` both answer in newline-delimited JSON, so one `stream` method
serves both. An error can also arrive mid-stream, after a 200. The iterator checks for that too.

In [ ]:
#| export
class OllamaError(RuntimeError):
    "What the daemon said went wrong, or that nothing answered."

def _err(r):
    "The daemon's own `error` message from a failed response, else the status line."
    try: msg = (r.json() or {}).get('error')
    except Exception: msg = None
    return msg or f'HTTP {r.status_code}: {(r.text or "")[:200]}'

class OllamaClient:
    "One Ollama daemon over HTTP: `/api/chat`, plus the model-management endpoints rishi drives."
    def __init__(self, host=None, timeout=600, client=None):
        self.url, self.timeout, self._own = ollama_url(host), timeout, client is None
        # a loopback daemon must not be sent through $HTTPS_PROXY, which is what `trust_env` would do
        self.client = client or httpx.Client(base_url=self.url, timeout=timeout,
                                             trust_env=not is_local(self.url))

    def __repr__(self): return f'OllamaClient({self.url})'

    def close(self):
        "Close the HTTP client, if this object made it. Idempotent."
        if self._own and getattr(self, 'client', None) is not None: self.client.close()
        self.client = None

    def _check(self, r):
        if r.status_code >= 400: raise OllamaError(_err(r))
        return r

    def get(self, path, **kw):
        "GET `path` as JSON."
        return self._check(self.client.get(path, **kw)).json()

    def post(self, path, payload, **kw):
        "POST `payload` to `path`, as JSON."
        return self._check(self.client.post(path, json=payload, **kw)).json()

    def stream(self, path, payload, timeout=None):
        "POST `payload`, yielding the daemon's newline-delimited JSON as it arrives."
        with self.client.stream('POST', path, json=payload, timeout=ifnone(timeout, self.timeout)) as r:
            if r.status_code >= 400:
                r.read(); raise OllamaError(_err(r))
            for line in r.iter_lines():
                if not (line := line.strip()): continue
                d = json.loads(line)
                # a stream can turn into an error after its 200, so this is not the check above again
                if (e := d.get('error')): raise OllamaError(e)
                yield d

    def version(self, timeout=5):
        "The daemon's version, or None when nothing answers at `url`."
        try: return self.get('/api/version', timeout=timeout).get('version')
        except Exception: return None

    def up(self, timeout=5):
        "Is a daemon answering at `url`?"
        return self.version(timeout) is not None

    def models(self):
        "Model names in the local store."
        return [m['name'] for m in (self.get('/api/tags').get('models') or [])]

    def have(self, model):
        "Is `model` downloaded already? Matches Ollama's implicit `:latest` tag."
        ms = self.models()
        return model in ms or f'{model}:latest' in ms

    def show(self, model):
        "What the daemon knows about `model`: `capabilities`, `template`, `model_info`."
        return self.post('/api/show', {'model': model})

    def ps(self):
        "Models currently loaded in memory."
        return self.get('/api/ps').get('models') or []

    def delete(self, model):
        "Remove `model` from the local store."
        self._check(self.client.request('DELETE', '/api/delete', json={'model': model}))
        return True

    def pull(self, model, on_progress=None):
        "Download `model`, handing each progress record to `on_progress`. Returns `model`."
        for d in self.stream('/api/pull', {'model': model, 'stream': True}):
            if on_progress: on_progress(d)
        return model

    def chat(self, payload, stream=False):
        "One `/api/chat` call: the whole response, or the chunk iterator when `stream` is set."
        if stream: return self.stream('/api/chat', {**payload, 'stream': True})
        return self.post('/api/chat', {**payload, 'stream': False})

`pull` hands on Ollama's own progress records. `pull_progress` renders them as one rewritten line,
so a 4GB download looks like a download rather than a hang.

In [ ]:
#| export
def fmt_bytes(n):
    "Bytes as a short human string."
    for u in ('B', 'KB', 'MB', 'GB'):
        if n < 1024 or u == 'GB': break
        n /= 1024
    return f'{n:.0f}B' if u == 'B' else f'{n:.1f}{u}'

def pull_progress(out=None):
    "A `pull` callback printing one rewritten line: the status, and how far the bytes have got."
    def show(d):
        s, done, total = d.get('status', ''), d.get('completed') or 0, d.get('total')
        line = f'{s} {fmt_bytes(done)}/{fmt_bytes(total)}' if total else s
        print(f'\r{line[:78]:<78}', end='', flush=True, file=out)
        if s in ('success', 'error'): print(file=out)
    return show

In [ ]:
# the client against a scripted daemon: one `httpx.MockTransport` stands in for the whole HTTP
# surface, so requests and responses are exercised rather than mocked away
def mk_client(handler, **kw):
    "An `OllamaClient` whose requests are answered by `handler` instead of a daemon."
    return OllamaClient(client=httpx.Client(transport=httpx.MockTransport(handler),
                                            base_url='http://test'), **kw)

seen = []
def handler(req):
    seen.append((req.method, req.url.path, json.loads(req.content) if req.content else {}))
    if req.url.path == '/api/version': return httpx.Response(200, json={'version': '0.12.0'})
    if req.url.path == '/api/tags':    return httpx.Response(200, json={'models': [{'name': 'qwen3:0.6b'}]})
    if req.url.path == '/api/show':    return httpx.Response(200, json={'capabilities': ['completion','tools','vision']})
    if req.url.path == '/api/ps':      return httpx.Response(200, json={'models': [{'name': 'qwen3:0.6b'}]})
    if req.url.path == '/api/delete':  return httpx.Response(200, text='')
    return httpx.Response(404, json={'error': f'unknown path {req.url.path}'})

cl = mk_client(handler)
test_eq(cl.version(), '0.12.0')
test_eq(cl.up(), True)
test_eq(cl.models(), ['qwen3:0.6b'])
test_eq(cl.have('qwen3:0.6b'), True)
test_eq(cl.have('gemma3:1b'), False)
test_eq(cl.show('qwen3:0.6b')['capabilities'], ['completion', 'tools', 'vision'])
test_eq(len(cl.ps()), 1)
test_eq(cl.delete('qwen3:0.6b'), True)
test_eq(seen[-1][:2], ('DELETE', '/api/delete'))
test_fail(lambda: cl.get('/api/nope'), contains='unknown path')      # the daemon's own words

In [ ]:
# a daemon that is not there reads as `None`, not as an exception; and an error arriving after a
# 200, part way down a stream, is raised from the iterator rather than swallowed
def dead(req): raise httpx.ConnectError('nothing listening')
test_eq(mk_client(dead).version(), None)
test_eq(mk_client(dead).up(), False)

lines = [{'status': 'pulling manifest'}, {'status': 'downloading', 'completed': 512, 'total': 1024},
         {'error': 'model not found'}]
def puller(req): return httpx.Response(200, text='\n'.join(json.dumps(d) for d in lines))
got = []
test_fail(lambda: mk_client(puller).pull('nope', got.append), contains='model not found')
test_eq([d['status'] for d in got], ['pulling manifest', 'downloading'])   # progress up to the failure

import io
buf = io.StringIO()
pull_progress(buf)({'status': 'downloading', 'completed': 512*1024, 'total': 1024*1024})
assert 'downloading 512.0KB/1.0MB' in buf.getvalue()
test_eq(fmt_bytes(900), '900B')
test_eq(fmt_bytes(1536), '1.5KB')

## Installing Ollama

Ollama ships one release archive per platform, and nothing in it needs root or a package manager.
The published one-liner pipes into `/usr`. The archive extracts anywhere. rishi keeps its own copy
under `~/.cache/rishi/ollama` and leaves the system alone. A machine with `ollama` on `$PATH`
keeps using that one.

Linux archives are zstd. That is stdlib from Python 3.14, and before it `rishi[ollama]` brings
`zstandard`. macOS ships gzip and Windows a zip.

In [ ]:
#| export
#: Where rishi installs its own copy of Ollama, when the machine has none.
OLLAMA_DIR = Path.home()/'.cache'/'rishi'/'ollama'

def install_url(system=None, machine=None):
    "URL of the Ollama release archive for this platform."
    s, m = (system or platform.system()).lower(), (machine or platform.machine()).lower()
    if s == 'darwin':  return 'https://ollama.com/download/ollama-darwin.tgz'
    if s == 'windows': return 'https://ollama.com/download/ollama-windows-amd64.zip'
    if s == 'linux':
        return f"https://ollama.com/download/ollama-linux-{'arm64' if m in ('arm64', 'aarch64') else 'amd64'}.tar.zst"
    raise OllamaError(f'no Ollama release for {s}/{m}; install it from https://ollama.com/download')

def ollama_bin(dir=None, system=True):
    "An `ollama` binary: `$OLLAMA_BIN`, then rishi's own install, then `$PATH` unless `system=False`. None if there is none."
    exe = 'ollama.exe' if platform.system() == 'Windows' else 'ollama'
    if (b := os.getenv('OLLAMA_BIN')) and Path(b).exists(): return Path(b)
    d = Path(dir or OLLAMA_DIR)
    if (p := first((d/'bin'/exe, d/exe), Path.exists)): return p
    return Path(w) if system and (w := shutil.which('ollama')) else None

In [ ]:
#| export
def _zstd_reader(f):
    "A decompressing reader over zstd file `f`: stdlib on Python 3.14+, else the `zstandard` package."
    try:
        from compression.zstd import ZstdFile
        return ZstdFile(f)
    except ImportError: pass
    try: from zstandard import ZstdDecompressor
    except ImportError: raise OllamaError(
        "the Linux Ollama archive is zstd-compressed, which needs Python 3.14+ or the zstandard "
        "package: pip install 'rishi[ollama]'") from None
    return ZstdDecompressor().stream_reader(f)

def _untar(t, dest):
    "Extract tarfile `t`, asking for the `tar` filter where the runtime has PEP 706 filters."
    try: t.extractall(dest, filter='tar')
    except TypeError: t.extractall(dest)

def extract_archive(src, dest):
    "Extract a `.tgz`, `.tar.zst` or `.zip` release archive into `dest`."
    dest, src = Path(dest), str(src)
    dest.mkdir(parents=True, exist_ok=True)
    if src.endswith('.zip'):
        with zipfile.ZipFile(src) as z: z.extractall(dest)
    elif src.endswith('.zst'):
        # not seekable: `r|` reads the tar forwards, as the decompressor hands it over
        with open(src, 'rb') as f, _zstd_reader(f) as r, tarfile.open(fileobj=r, mode='r|') as t:
            _untar(t, dest)
    else:
        with tarfile.open(src, 'r:gz') as t: _untar(t, dest)
    return dest

def download(url, dest, on_progress=None, timeout=600):
    "Stream `url` to `dest`, reporting `(done, total)` bytes to `on_progress`."
    with httpx.stream('GET', url, follow_redirects=True, timeout=timeout) as r:
        r.raise_for_status()
        total, done = int(r.headers.get('content-length') or 0), 0
        with open(dest, 'wb') as f:
            for chunk in r.iter_bytes(1<<20):
                f.write(chunk); done += len(chunk)
                if on_progress: on_progress(done, total)
    return Path(dest)

def install_ollama(dir=OLLAMA_DIR, url=None, force=False, on_progress=None):
    "Install Ollama under `dir` for this user: no root, no package manager, no system service. Returns the binary."
    if not force and (b := ollama_bin(dir, system=False)): return b
    url, dir = url or install_url(), Path(dir)
    dir.mkdir(parents=True, exist_ok=True)
    tmp = dir/('rishi-download' + ''.join(Path(urlsplit(url).path).suffixes))
    try:
        download(url, tmp, on_progress)
        extract_archive(tmp, dir)
    finally: tmp.unlink(missing_ok=True)
    if not (b := ollama_bin(dir, system=False)):
        raise OllamaError(f'no ollama binary under {dir} after extracting {url}')
    b.chmod(0o755)
    return b

In [ ]:
test_eq(install_url('Linux', 'x86_64'),  'https://ollama.com/download/ollama-linux-amd64.tar.zst')
test_eq(install_url('Linux', 'aarch64'), 'https://ollama.com/download/ollama-linux-arm64.tar.zst')
test_eq(install_url('Darwin', 'arm64'),  'https://ollama.com/download/ollama-darwin.tgz')
test_eq(install_url('Windows', 'amd64'), 'https://ollama.com/download/ollama-windows-amd64.zip')
test_fail(lambda: install_url('Plan9', 'x86_64'), contains='no Ollama release')

In [ ]:
# extraction and binary discovery, on a tarball built here. `bin/ollama` is what the real Linux
# and macOS archives lay down, and what `ollama_bin` has to find without consulting $PATH
import tempfile

with tempfile.TemporaryDirectory() as d:
    d = Path(d)
    (d/'bin').mkdir()
    (d/'bin'/'ollama').write_text('#!/bin/sh\necho fake')
    with tarfile.open(d/'a.tgz', 'w:gz') as t: t.add(d/'bin'/'ollama', arcname='bin/ollama')
    out = extract_archive(d/'a.tgz', d/'out')
    assert (out/'bin'/'ollama').exists()
    test_eq(ollama_bin(out, system=False), out/'bin'/'ollama')
    test_eq(install_ollama(out), out/'bin'/'ollama')       # already there, so nothing is downloaded
    test_eq(ollama_bin(d/'empty', system=False), None)

    # a flat archive (just `ollama`) is found too, and $OLLAMA_BIN wins over both
    (d/'flat').mkdir(); (d/'flat'/'ollama').write_text('x')
    test_eq(ollama_bin(d/'flat', system=False), d/'flat'/'ollama')
    os.environ['OLLAMA_BIN'] = str(d/'bin'/'ollama')
    try: test_eq(ollama_bin(d/'flat', system=False), d/'bin'/'ollama')
    finally: del os.environ['OLLAMA_BIN']

## Running the daemon

`OllamaServer` starts `ollama serve` and waits for it to answer, installing Ollama first if the
machine has none.

Ollama configures some things per process rather than per request. Those are constructor arguments
here. They cover the model store, the default context length, how long a model stays resident,
flash attention, KV cache quantization, and the parallel and loaded-model limits.

A daemon already listening is left alone. Someone else's server is not rishi's to restart or
reconfigure. One rishi starts is stopped at interpreter exit, so a notebook leaks nothing.

In [ ]:
#| export
#: Constructor argument -> the environment variable `ollama serve` reads it from.
SERVE_ENV = {'models': 'OLLAMA_MODELS', 'n_ctx': 'OLLAMA_CONTEXT_LENGTH', 'keep_alive': 'OLLAMA_KEEP_ALIVE',
             'flash_attn': 'OLLAMA_FLASH_ATTENTION', 'kv_cache_type': 'OLLAMA_KV_CACHE_TYPE',
             'num_parallel': 'OLLAMA_NUM_PARALLEL', 'max_loaded': 'OLLAMA_MAX_LOADED_MODELS'}

class OllamaServer:
    "An `ollama serve` process rishi starts and owns, installing Ollama first if the machine has none."
    def __init__(self, host=None, bin=None, dir=OLLAMA_DIR, models=None, n_ctx=None, keep_alive=None,
                 flash_attn=None, kv_cache_type=None, num_parallel=None, max_loaded=None, log=None):
        self.url, self.proc = ollama_url(host), None
        store_attr('bin,dir,models,n_ctx,keep_alive,flash_attn,kv_cache_type,num_parallel,max_loaded,log')

    def __repr__(self): return f'OllamaServer({self.url}, running={self.running})'

    @property
    def running(self):
        "Is the process this object started still alive?"
        return self.proc is not None and self.proc.poll() is None

    def env(self):
        "The environment `ollama serve` gets: this process's own, plus what this server configures."
        e = dict(os.environ, OLLAMA_HOST=self.url)
        for k, var in SERVE_ENV.items():
            if (v := getattr(self, k)) is not None: e[var] = str(int(v) if isinstance(v, bool) else v)
        return e

    def client(self, **kw):
        "An `OllamaClient` pointed at this server."
        return OllamaClient(self.url, **kw)

    def start(self, install=True, timeout=60, on_progress=None):
        "Start the daemon and wait for it to answer. A daemon already listening at `url` is left alone."
        cl = self.client(timeout=30)
        try:
            if cl.up(): return self
            if (b := self.bin) is not None:
                # an explicit `bin` that is not there is a mistake to report, not one to route around
                if not Path(b).exists(): raise OllamaError(f'no ollama binary at {b}')
            elif not (b := ollama_bin(self.dir)):
                if not install: raise OllamaError(
                    'no ollama binary found; install one, or call install_ollama()')
                b = install_ollama(self.dir, on_progress=on_progress)
            self.bin = b
            out = open(self.log, 'ab') if self.log else subprocess.DEVNULL
            # its own session, so a Ctrl-C meant for the parent does not also kill the daemon
            self.proc = subprocess.Popen([str(b), 'serve'], env=self.env(), stdout=out, stderr=out,
                                         start_new_session=True)
            atexit.register(self.stop)
            for _ in range(int(timeout*10)):
                if cl.up(timeout=2): return self
                if not self.running: raise OllamaError(
                    f'`{b} serve` exited with {self.proc.returncode}' + (f'; see {self.log}' if self.log else ''))
                time.sleep(0.1)
            self.stop()
            raise OllamaError(f'`{b} serve` did not answer at {self.url} within {timeout}s')
        finally: cl.close()

    def stop(self, timeout=10):
        "Stop the daemon, if this object started it. Idempotent, and registered to run at exit."
        p, self.proc = self.proc, None
        if p is None or p.poll() is not None: return self
        p.terminate()
        try: p.wait(timeout)
        except subprocess.TimeoutExpired: p.kill(); p.wait()
        return self

In [ ]:
#| export
#: Daemons rishi started, by URL, so a second chat reuses the first one's server.
_servers = {}

def ensure_ollama(host=None, install=True, serve=True, timeout=60, on_progress=None, srv_kw=None):
    "A client for a live daemon at `host`: reuse one already listening, else start one, installing Ollama if needed."
    url = ollama_url(host)
    cl = OllamaClient(url)
    if cl.up(): return cl
    cl.close()
    if not serve: raise OllamaError(
        f'nothing is listening at {url}. Start one with `ollama serve`, or pass serve=True.')
    srv = _servers.get(url) or OllamaServer(url, **(srv_kw or {}))
    srv.start(install=install, timeout=timeout, on_progress=on_progress)
    _servers[url] = srv
    return srv.client()

def stop_ollama(host=None):
    "Stop the daemon rishi started at `host`, or every one of them when `host` is None."
    for url in ([ollama_url(host)] if host else list(_servers)):
        if (s := _servers.pop(url, None)): s.stop()

In [ ]:
# the process-level configuration, which is the half of Ollama that is not a request field
s = OllamaServer('localhost:11500', models='/data/models', n_ctx=8192, keep_alive='30m',
                 flash_attn=True, kv_cache_type='q8_0', num_parallel=2)
e = s.env()
test_eq(e['OLLAMA_HOST'], 'http://localhost:11500')
test_eq(e['OLLAMA_MODELS'], '/data/models')
test_eq(e['OLLAMA_CONTEXT_LENGTH'], '8192')
test_eq(e['OLLAMA_KEEP_ALIVE'], '30m')
test_eq(e['OLLAMA_FLASH_ATTENTION'], '1')            # a bool, in the spelling Ollama reads
test_eq(e['OLLAMA_KV_CACHE_TYPE'], 'q8_0')
test_eq(e['OLLAMA_NUM_PARALLEL'], '2')
assert 'OLLAMA_MAX_LOADED_MODELS' not in e           # unset stays unset, rather than guessing a default
test_eq(s.running, False)
test_eq(s.stop().running, False)                     # stopping one that never started is fine

In [ ]:
# with nothing listening and no leave to start anything, the error names both ways out
test_fail(lambda: ensure_ollama('127.0.0.1:11599', serve=False), contains='ollama serve')
test_fail(lambda: OllamaServer('127.0.0.1:11599', bin='/nonexistent/ollama').start(install=False, timeout=1),
          contains='no ollama binary')

## Messages

Ollama's message shape is OpenAI's, with two differences. Images sit in a sibling `images` list of
bare base64 rather than in content parts. Thinking has its own `thinking` field rather than `<think>`
tags in the text. `to_ollama_msg` converts rishi's canonical history to that, and carries assistant
thinking back out so a resumed conversation keeps it.

Images go out on every turn rather than collapsing to a placeholder, as on `remote` and unlike
`llama`. The daemon holds the KV cache, so a repeat costs a prefix match rather than a re-encode.

Audio has nowhere to go here. `rishi.llama` (mtmd) and `rishi.mlx` both take it, and the error says
so rather than dropping the clip.

In [ ]:
#| export
def _img_b64(p):
    "The bare base64 of an OpenAI-style image part, with any data-URL wrapper taken off."
    u = p.get('image_url')
    u = u.get('url', '') if isinstance(u, dict) else (u or '')
    return u.split(',', 1)[1] if u.startswith('data:') else u

def to_ollama_msg(m):
    "One canonical rishi history dict -> an Ollama chat message."
    if (role := m.get('role', 'user')) == 'tool':
        return {'role': 'tool', 'content': str(m.get('content', '')), 'tool_name': m.get('name', '')}
    c = m.get('content')
    txt, imgs = [], []
    for p in (c if isinstance(c, list) else [{'type': 'text', 'text': c or ''}]):
        if not isinstance(p, dict): continue
        if (t := p.get('type')) == 'text': txt.append(p.get('text', ''))
        elif t == 'image_url': imgs.append(_img_b64(p))
        elif t == 'input_audio': raise TypeError(
            "Ollama's chat API carries text and images only. For audio use runtime='llama' (mtmd) "
            "or runtime='mlx'.")
    out = {'role': role, 'content': '\n'.join(x for x in txt if x)}
    if imgs: out['images'] = imgs
    if (th := (m.get('channels') or {}).get('thought')): out['thinking'] = th
    if (tcs := m.get('tool_calls')):
        out['tool_calls'] = [{'function': {'name': tc_name(tc),
                                           'arguments': (tc.get('function') or {}).get('arguments') or {}}}
                             for tc in tcs]
    return out

def to_ollama_msgs(msgs, sp=''):
    "Canonical rishi history -> Ollama messages, with `sp` as the leading system message."
    sys = [{'role': 'system', 'content': sp}] if sp else []
    return sys + [to_ollama_msg(m) for m in listify(msgs) if m.get('role') != 'system']

A reply carries thinking in `thinking` and calls in `tool_calls`. A model whose template was built
for neither writes `<think>` and `<tool_call>` into the text instead. `norm_ochat` reads both, as
`rishi.core.norm_resp` does for llama.cpp, so an arbitrary GGUF off the Hub behaves like a model
from Ollama's own library.

Usage comes from the eval counts. Ollama reports no reuse figure, so `use.cached_tokens` reads zero
on a conversation whose prefix the daemon is in fact reusing.

In [ ]:
#| export
def ollama_usage(r, model=None):
    "Ollama's eval counts -> a rishi usage dict."
    p, c = r.get('prompt_eval_count') or 0, r.get('eval_count') or 0
    return {'prompt_tokens': p, 'completion_tokens': c, 'total_tokens': p + c,
            'model': model or r.get('model')}

def _tcs(tcs):
    "Ollama tool calls -> canonical `ToolCall`s. Ollama sends arguments as an object, and no call id."
    return [ToolCall(name=(f := tc.get('function') or {}).get('name', ''),
                     arguments=parse_args(f.get('arguments'))) for tc in (tcs or [])]

def _tag_tcs(tcs):
    "`<tool_call>` blocks parsed out of reply text -> canonical `ToolCall`s."
    return [ToolCall(name=tc['function']['name'], arguments=tc['function']['arguments'], id=tc['id'])
            for tc in tcs]

def norm_ochat(r, model=None):
    "One `/api/chat` response -> a rishi `Resp`, reading `<think>` and `<tool_call>` text as well as the native fields."
    m = r.get('message') or {}
    text, th = split_think(m.get('content') or '')
    text, tag_tcs = parse_tool_tags(text)
    th = '\n'.join(x for x in (m.get('thinking') or '', th) if x)
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if (tcs := _tcs(m.get('tool_calls')) + _tag_tcs(tag_tcs)): res['tool_calls'] = tcs
    if r.get('done_reason') == 'length': res['truncated'] = True
    res['usage'] = ollama_usage(r, model)
    return Resp(res)

In [ ]:
# text, images and thinking, each into the field Ollama reads it from
test_eq(to_ollama_msg({'role': 'user', 'content': 'hi'}), {'role': 'user', 'content': 'hi'})
m = to_ollama_msg(mk_msg(['what is this?', Path('images.jpeg').read_bytes()]))
test_eq(m['content'], 'what is this?')
test_eq(len(m['images']), 1)
assert not m['images'][0].startswith('data:')          # bare base64, not a data URL
test_eq(to_ollama_msg({'role': 'assistant', 'content': 'a', 'channels': {'thought': 't'}})['thinking'], 't')

# a tool call goes out as an object, and its result comes back named
call_msg = {'role': 'assistant', 'content': '', 'tool_calls': [ToolCall(name='add', arguments={'a': 1, 'b': 2})]}
test_eq(to_ollama_msg(call_msg)['tool_calls'], [{'function': {'name': 'add', 'arguments': {'a': 1, 'b': 2}}}])
test_eq(to_ollama_msg({'role': 'tool', 'name': 'add', 'content': '3'}),
        {'role': 'tool', 'content': '3', 'tool_name': 'add'})

# the system prompt leads, and a system message already in `hist` is dropped rather than duplicated
ms = to_ollama_msgs([{'role': 'system', 'content': 'old'}, {'role': 'user', 'content': 'hi'}], sp='be brief')
test_eq(ms, [{'role': 'system', 'content': 'be brief'}, {'role': 'user', 'content': 'hi'}])

test_fail(lambda: to_ollama_msg(mk_msg([Path('speech.wav').read_bytes(), 'transcribe'])),
          contains="runtime='llama'")

In [ ]:
# the native fields
r = norm_ochat({'model': 'qwen3:0.6b', 'done_reason': 'stop',
                'message': {'role': 'assistant', 'content': 'hello', 'thinking': 'thinking hard'},
                'prompt_eval_count': 10, 'eval_count': 4})
test_eq(resp_text(r), 'hello')
test_eq(thought(r), 'thinking hard')
test_eq(r['usage'], {'prompt_tokens': 10, 'completion_tokens': 4, 'total_tokens': 14, 'model': 'qwen3:0.6b'})
assert 'truncated' not in r
test_eq(truncated(norm_ochat({'message': {'content': 'x'}, 'done_reason': 'length'})), True)

# a native tool call: arguments as an object, and an id invented because Ollama sends none
r = norm_ochat({'message': {'tool_calls': [{'function': {'name': 'add', 'arguments': {'a': 1}}}]}})
test_eq(len(r['tool_calls']), 1)
test_eq(r['tool_calls'][0].name, 'add')
test_eq(r['tool_calls'][0].arguments, {'a': 1})
assert r['tool_calls'][0]['id']

# and a model that writes tags into the text instead, as an arbitrary GGUF off the Hub will
r = norm_ochat({'message': {'content': '<think>hmm</think>ok\n<tool_call>{"name": "add", "arguments": {"a": 1}}</tool_call>'}})
test_eq(resp_text(r), 'ok')
test_eq(thought(r), 'hmm')
test_eq(r['tool_calls'][0].name, 'add')

## OllamaChat

The tool loop, approval, budget, callbacks and context handling are `rishi.core`'s, as on every
backend. This class adds the request, and the daemon it goes to.

- `think` is a request field here, not a system-prompt trick: `True`, `False`, or one of `'low'`,
  `'medium'`, `'high'` and `'max'`. A model with no thinking refuses the field. rishi then drops it,
  retries once, and stops sending it.
- `n_ctx` and `n_gpu_layers` keep their `rishi.llama` names and go out as `num_ctx` and `num_gpu`.
  Anything else Ollama samples with goes in `options=`.
- `structured` uses Ollama's own `format`, so the JSON is schema-constrained rather than asked for.
- `pull=True` downloads a missing model on construction, as the llama backend downloads a missing
  GGUF.
- `ctx_limit` follows `n_ctx`, else Ollama's own default. `pct_full` and `SlidingWindowCallback`
  need it to be right, and the daemon defaults to 4096 whatever the model was trained for.

In [ ]:
#| export
def dflt_ctx():
    "The context length the daemon uses when a request names none: `$OLLAMA_CONTEXT_LENGTH`, else Ollama's own 4096."
    try: return int(os.getenv('OLLAMA_CONTEXT_LENGTH') or 4096)
    except ValueError: return 4096

class OllamaChat(ToolLoopMixin, Chat):
    "Sync chat against an Ollama daemon through Urai's tool loop."
    _runtime = 'ollama'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    mk_content, mk_msg, mk_msgs = staticmethod(mk_content), staticmethod(mk_msg), staticmethod(mk_msgs)

    @staticmethod
    def fmt2hist(msgs):
        "Normalize incoming messages to canonical rishi history dicts."
        return mk_msgs(msgs)
    @staticmethod
    def hist2fmt(msgs):
        "Canonical rishi history dicts -> Ollama messages."
        return [to_ollama_msg(m) for m in listify(msgs)]

    #: what Ollama calls each portable option, inside its `options` block
    _opt_map = {'ctx': 'num_ctx', 'temp': 'temperature', 'max_output_tokens': 'num_predict'}
    _opt_skip = ('effort', 'tool_mode', 'api_key', 'base_url')   # nothing local takes these

    def __init__(self, model=None, *, runtime=None, model_path=None, opts=None,
                 host=None, client=None,   # a daemon to use instead of finding or starting one
                 quant='Q4_K_M',           # which quantization to name when pulling
                 pull=True, install=True, serve=True, srv_kw=None, on_progress=None,
                 n_gpu_layers=None,
                 keep_alive=None,          # how long the daemon holds the weights after a turn
                 options=None,             # Ollama's own `options` block, merged last
                 comp_kw=None,             # merged into the request body verbatim
                 **kw):                    # portable options; see `urai.ChatOpts`
        o = ChatOpts.create(opts, **kw)
        model = split_runtime(model)[1]
        self.model_id = ollama_model(model or model_path, quant)
        self._own_client = client is None
        self.client = client or ensure_ollama(host, install=install, serve=serve, srv_kw=srv_kw,
                                              on_progress=on_progress)
        self._opts = {k: v for k, v in dict(temperature=o.temp, top_k=o.top_k, top_p=o.top_p,
                                            seed=o.seed, num_ctx=o.ctx or None,
                                            num_gpu=n_gpu_layers).items() if v is not None}
        self._opts.update(options or {})
        self.think, self.keep_alive = o.think, keep_alive
        self.max_output_tokens = o.max_output_tokens or None
        # a model with no thinking refuses `think`; this goes false on the first refusal
        self._think_ok, self.comp_kw, self._ctx_tokens = o.think is not None, comp_kw or {}, 0
        self._set_tools(o.tools)
        if pull and not self.client.have(self.model_id):
            self.client.pull(self.model_id, ifnone(on_progress, pull_progress()))
        self.ctx_limit = o.ctx or dflt_ctx()
        self._setup(model, o)

    def close(self):
        "Release this chat's HTTP client. The daemon is shared, so it stays up; `stop_ollama()` ends one rishi started."
        if self._own_client and getattr(self, 'client', None) is not None: self.client.close()
        self.client = None

    def _note_usage(self, res):
        "Keep the last turn's counts. Reported for cost, never for occupancy: see `token_count`."
        n = ((res or {}).get('usage') or {}).get('total_tokens') or 0
        if n: self._ctx_tokens = int(n)
        return res

    @property
    def token_count(self):
        "What the conversation holds, estimated from it."
        return est_tokens(render_prompt(self.hist)) + est_tokens(self.sp or '')

    def count_tokens(self, text):
        "Estimated tokens in `text`. Ollama exposes no tokenizer, so the real count only arrives in `use`, after a turn."
        return est_tokens(text)

    def unload(self):
        "Drop the model from the daemon's memory now, rather than when `keep_alive` runs out."
        self.client.chat({'model': self.model_id, 'messages': [], 'keep_alive': 0})
        return self

    def _msgs(self):
        "The whole conversation as Ollama messages, system prompt first."
        return to_ollama_msgs(self.hist, self.sp)

    def _payload(self, stream=False, msgs=None, tools=True, fmt=None, think=..., **turn):
        "One `/api/chat` request body, with this turn's own generation options applied."
        t = self.map_opts(turn)
        if (th := t.pop('think', ...)) is not ...: think = th   # top-level, not an option
        n = ifnone(t.pop('num_predict', None), self.max_output_tokens)
        opts = {**self._opts, **t}
        if n is not None: opts['num_predict'] = int(n)
        p = {'model': self.model_id, 'messages': ifnone(msgs, self._msgs()), 'stream': stream}
        if opts: p['options'] = opts
        if tools and self.toolspecs: p['tools'] = self.toolspecs
        if fmt is not None: p['format'] = fmt
        if self.keep_alive is not None: p['keep_alive'] = self.keep_alive
        if (th := self.think if think is ... else think) is not None and self._think_ok: p['think'] = th
        return {**p, **self.comp_kw}

    def _drop_think(self, payload, e):
        "Was `e` the daemon refusing `think`? Then take it off `payload` and stop sending it."
        if 'think' not in payload or 'think' not in str(e).lower(): return False
        self._think_ok = False
        payload.pop('think')
        return True

    def _ask(self, payload):
        "One request, retried once without `think` if the model has no thinking to give."
        try: return self.client.chat(payload)
        except OllamaError as e:
            if not self._drop_think(payload, e): raise
            return self.client.chat(payload)

    def _ask_stream(self, payload):
        "Stream one request. A refusal only lands on the first chunk, so the retry has to sit there."
        it = iter(self.client.chat(payload, stream=True))
        try: first_chunk = next(it)
        except StopIteration: return
        except OllamaError as e:
            if not self._drop_think(payload, e): raise
            it = iter(self.client.chat(payload, stream=True))
            first_chunk = next(it)
        yield first_chunk
        yield from it

    def _model_step(self, **kw):
        "One completion, normalized to a `Resp`. The single wire call `ToolLoopMixin` drives."
        return self._note_usage(norm_ochat(self._ask(self._payload(**kw)), self.model_id))

    def _stream_step(self, **kw):
        "Stream one completion, yielding chunk dicts and leaving the merged `Resp` on `self._step_res`."
        split, tcs, native_th, last = StreamSplit(), [], '', {}
        for d in self._ask_stream(self._payload(stream=True, **kw)):
            msg = d.get('message') or {}
            if (th := msg.get('thinking')):
                native_th += th
                yield {'channels': {'thought': th}}
            if (c := msg.get('content')): yield from split.feed(c)
            tcs += msg.get('tool_calls') or []
            if d.get('done'): last = d
        yield from split.finish()
        res = {'role': 'assistant', 'content': split.text}
        if (th := '\n'.join(x for x in (native_th, split.thought) if x)): res['channels'] = {'thought': th}
        if (calls := _tcs(tcs) + _tag_tcs(split.tool_calls)): res['tool_calls'] = calls
        if last.get('done_reason') == 'length': res['truncated'] = True
        res['usage'] = ollama_usage(last, self.model_id)
        self._step_res = self._note_usage(Resp(res))

    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        "Stateless completion text: no history, no tools. It shares the daemon, which caches per conversation."
        msgs = ([{'role': 'system', 'content': sp}] if sp else []) + [{'role': 'user', 'content': prompt}]
        p = self._payload(msgs=msgs, tools=False, think=think, max_output_tokens=max_tokens)
        return resp_text(norm_ochat(self._ask(p), self.model_id))

    def _structured_call(self, prompt, schema, sp):
        "Schema-constrained JSON through Ollama's own `format`, returning the decoded argument dict."
        msgs = ([{'role': 'system', 'content': sp}] if sp else []) + [{'role': 'user', 'content': prompt}]
        p = self._payload(msgs=msgs, tools=False, fmt=get_schema(schema)['input_schema'])
        txt = resp_text(norm_ochat(self._ask(p), self.model_id))
        try: return json.loads(txt)
        except json.JSONDecodeError:
            raise ValueError(f'model returned no JSON for the schema; reply: {txt[:200]!r}') from None

`/api/show` reports what a model can do, which beats every guess `rishi.core.model_caps` makes
elsewhere: the daemon has the weights. `model_caps(..., runtime='ollama')` asks it, and falls back
to text-only when no daemon answers.

In [ ]:
#| export
#: `/api/show` capability -> Urai input modality.
CAP_INPUT = {'vision': 'image'}

def ollama_caps(model=None, client=None, quant='Q4_K_M'):
    "What an Ollama model accepts, from the daemon's own `/api/show`. None when no daemon answers."
    try:
        cl = client or OllamaClient()
        if not cl.up(): return None
        caps = cl.show(ollama_model(model, quant)).get('capabilities') or []
    except Exception: return None
    return Caps(('text',) + tuple(dict.fromkeys(CAP_INPUT[c] for c in caps if c in CAP_INPUT)),
                ('text',), (), 'runtime')

### Against a real daemon

Everything above runs offline, against a scripted daemon. The cells below need Ollama and download a
model on first use, so they are `#| eval: false` and CI skips them.

`OllamaChat` on a bare machine installs Ollama under `~/.cache/rishi/ollama`, starts it, pulls the
model, and answers.

In [ ]:
#| eval: false
chat = Chat('ollama/qwen3:0.6b', think=False, sp='You are concise.')
print(resp_text(chat('Give me one fact about lobsters.')))
print(chat.use)

In [ ]:
#| eval: false
# thinking levels, tools and structured output, on the real thing
t = Chat('ollama/qwen3:0.6b', tools=[add], think='low')
print(resp_text(t('What is 2 + 3? Use the add tool.')))
print(t.structured('Extract: John Smith is 30.', Person))
print(t.classify('I loved this film!', ['positive', 'negative']))
t.close()

In [ ]:
#| eval: false
# the daemon, driven from Python: what is downloaded, what is loaded, what a model can do
cl = ensure_ollama()
print(cl.version(), cl.models())
print([m['name'] for m in cl.ps()])
print(model_caps('qwen3:0.6b', runtime='ollama'))
print(model_caps('gemma3:4b', runtime='ollama'))     # vision, per the daemon itself

In [ ]:
#| eval: false
# a hub GGUF, addressed as `rishi.llama` addresses it, served by Ollama instead
g = Chat('Qwen/Qwen3-0.6B-GGUF', runtime='ollama', quant='Q4_K_M')
print(g.model_id)                                    # hf.co/Qwen/Qwen3-0.6B-GGUF:Q4_K_M
print(resp_text(g('Say hello in one short sentence.')))
g.unload(); g.close()

## What this backend does not do

- **Audio.** Ollama's chat API carries images, not audio. `rishi.llama` (mtmd) and `rishi.mlx` take
  it.
- **A saved KV cache.** `rishi.llama` and `rishi.mlx` write theirs to disk and read it back. Ollama
  keeps its cache inside the daemon and offers no handle on it, so `use.cached_tokens` reads zero
  even while the daemon reuses the prefix.
- **Exact token counts before a turn.** There is no tokenizer endpoint, so `count_tokens` estimates.
  The real count arrives with the reply, in `use`.
- **Local `.gguf` paths.** Ollama serves what is in its own store. `runtime='llama'` reads a file.
- **A broker.** `LlamaBroker` exists because one `Llama` belongs to one process. The daemon already
  is that. Size it with `OLLAMA_NUM_PARALLEL` and `OLLAMA_MAX_LOADED_MODELS`.

### Tests

The tests above cover host parsing, model ids, the HTTP surface, the installer, the server
environment, message conversion and response normalization. The backend itself runs against an
`httpx.MockTransport` daemon: tools, approval, thinking and its refusal, options, structured output,
streaming, and a history hand-off. None of it needs Ollama, so CI runs all of it.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()